# Build a small demo db

Seeds `cache/protein_selector.db` with a handful of **hand-authored, real**
PDB entries (4HHB/1UBQ/1STP) so `db_explorer.ipynb` has something to show
immediately -- no network calls, no conda env, nothing slow. This is NOT the
real pipeline (`protein_selector.pipeline.run_pipeline`) -- it writes
directly to each domain's persisted tables via the same `upsert_*`
functions the real pipeline/validators use, so the shape is real, but the
values are illustrative, not fetched.

Re-runnable: every table is upsert-based, so running this again just
refreshes the same three rows rather than duplicating them.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [ ]:
from pathlib import Path

from protein_selector.bioinformatics.store import upsert_literature_counts
from protein_selector.core.db import DEFAULT_DB_PATH
from protein_selector.core.validation_result import (
    FailureMode,
    ValidationResult,
    ValidationStatus,
)
from protein_selector.core.validation_store import upsert_validation_results
from protein_selector.docking.parameterizability import ParameterizabilityResult
from protein_selector.docking.pocket import PocketDetectionResult, PocketInfo
from protein_selector.docking.store import (
    upsert_ligand_ccd_codes,
    upsert_parameterizability,
    upsert_pocket_detection,
)
from protein_selector.modeling.alphafold_lookup import AlphaFoldEntry
from protein_selector.modeling.store import upsert_alphafold_entry
from protein_selector.structural_biology.candidates import CandidateEntry
from protein_selector.structural_biology.simulability import SimulabilityResult
from protein_selector.structural_biology.store import upsert_candidates, upsert_simulability

# Same repo-root-vs-notebooks/ ambiguity as db_explorer.ipynb -- write next
# to wherever a cache/ dir already exists, defaulting to repo-root-relative
# (i.e. run this from a kernel launched at the repo root, or adjust below).
DB_PATH = DEFAULT_DB_PATH
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
DB_PATH

## Candidates + simulability

Three real PDB entries, illustrative field values (not live-fetched).

In [ ]:
entries = [
    CandidateEntry(
        pdb_id="4HHB", title="Hemoglobin", organism="Homo sapiens",
        n_residues=141, n_atoms=1069, n_modeled_residues=141, n_unmodeled_residues=0,
        resolution=1.74, method="X-RAY DIFFRACTION", n_protein_entities=2,
        uniprot_ids=["P69905"], non_polymer_entity_ids=["4HHB_3"],
    ),
    CandidateEntry(
        pdb_id="1UBQ", title="Ubiquitin", organism="Homo sapiens",
        n_residues=76, n_atoms=602, n_modeled_residues=76, n_unmodeled_residues=0,
        resolution=1.8, method="X-RAY DIFFRACTION", n_protein_entities=1,
        uniprot_ids=["P0CG48"],
    ),
    CandidateEntry(
        pdb_id="1STP", title="Streptavidin", organism="Streptomyces avidinii",
        n_residues=127, n_atoms=980, n_modeled_residues=120, n_unmodeled_residues=7,
        resolution=2.6, method="X-RAY DIFFRACTION", n_protein_entities=1,
        uniprot_ids=["P22629"], non_polymer_entity_ids=["1STP_2"],
    ),
]
upsert_candidates(entries, db_path=DB_PATH)

upsert_simulability([
    SimulabilityResult(pdb_id="4HHB", passed=True, reasons=[]),
    SimulabilityResult(pdb_id="1UBQ", passed=True, reasons=[]),
    SimulabilityResult(pdb_id="1STP", passed=False, reasons=["resolution 2.6 above ceiling 2.5"]),
], db_path=DB_PATH)

## Ligands + parameterizability + pocket detection

In [ ]:
upsert_ligand_ccd_codes({"4HHB": ["HEM"], "1STP": ["BTN"]}, db_path=DB_PATH)

upsert_parameterizability([
    ParameterizabilityResult(ligand_id="HEM", passed=True, reasons=[]),
    ParameterizabilityResult(ligand_id="BTN", passed=True, reasons=[]),
], db_path=DB_PATH)

upsert_pocket_detection([
    PocketDetectionResult(
        pdb_id="4HHB", passed=True,
        pockets=[PocketInfo(pocket_number=1, druggability_score=0.72)],
    ),
    PocketDetectionResult(
        pdb_id="1STP", passed=True,
        pockets=[PocketInfo(pocket_number=1, druggability_score=0.91)],
    ),
], db_path=DB_PATH)

## Literature counts + AlphaFold DB entry

In [ ]:
upsert_literature_counts({"4HHB": 8213, "1UBQ": 452, "1STP": 301}, db_path=DB_PATH)

upsert_alphafold_entry(
    AlphaFoldEntry(
        uniprot_accession="P69905", entry_id="AF-P69905-F1", mean_plddt=98.06,
        fraction_plddt_very_low=0.0, fraction_plddt_low=0.007,
        fraction_plddt_confident=0.0, fraction_plddt_very_high=0.993,
        pdb_url="https://alphafold.ebi.ac.uk/files/AF-P69905-F1-model_v6.pdb",
        cif_url="https://alphafold.ebi.ac.uk/files/AF-P69905-F1-model_v6.cif",
        pae_doc_url="https://alphafold.ebi.ac.uk/files/AF-P69905-F1-predicted_aligned_error_v6.json",
        model_created_date="2025-08-01T00:00:00Z",
    ),
    db_path=DB_PATH,
)

## Validation results (ex02/ex03/ex04) -- one pass, one fail, one not-run per exercise

1UBQ passes ex03; 4HHB fails ex03 (a real failure mode: HEM left in, no force-field
template); 1STP passes ex04. Every other exercise/candidate combination is left
unset on purpose, to show `db_explorer.ipynb`'s `"not_run"` status too.

In [ ]:
upsert_validation_results(
    "ex02", [ValidationResult(pdb_id="4HHB", status=ValidationStatus.SUCCESS, effort_seconds=0.4)],
    db_path=DB_PATH,
)
upsert_validation_results(
    "ex03",
    [
        ValidationResult(pdb_id="1UBQ", status=ValidationStatus.SUCCESS, effort_seconds=23.0),
        ValidationResult(
            pdb_id="4HHB", status=ValidationStatus.FAILURE,
            failure_mode=FailureMode.PARAMETERIZATION,
            notes=["HEM left in, no force-field template found"],
        ),
    ],
    db_path=DB_PATH,
)
upsert_validation_results(
    "ex04", [ValidationResult(pdb_id="1STP", status=ValidationStatus.SUCCESS, effort_seconds=45.0)],
    db_path=DB_PATH,
)

print(f"demo db ready at {DB_PATH.resolve()}")

Now open `db_explorer.ipynb` (same kernel/cwd) to browse it.